# Native Indic G2P — r2 T4 training

Select **Runtime → Change runtime type → T4 GPU**, then run every cell in order. The dataset is already built; this notebook only verifies it and trains the model.

In [ ]:
%cd /content
!test -d /content/Native-indic-G2P/.git || git clone --depth 1 https://github.com/manu2407/Native-indic-G2P.git /content/Native-indic-G2P
!git -C /content/Native-indic-G2P pull --ff-only
%cd /content/Native-indic-G2P
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU runtime before training.'
print(torch.__version__, torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
import hashlib, tarfile
from pathlib import Path
from neural_data import verify_manifest

archive = Path('data/hindi_g2p_v2_1m_r2.tar.gz')
expected = 'f66de01174569ed7b9ed59ff6d150f3d766017980e4de1c0a54b3b0987641d17'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, 'Dataset archive checksum mismatch.'
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('.', filter='data')
manifest_path = Path('datasets/validation/neural_hindi_g2p_v2_1m_r2/manifest.json')
manifest = verify_manifest(manifest_path)
print({name: manifest[name]['records'] for name in ('training', 'dev', 'blind')})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
checkpoint = Path('/content/drive/MyDrive/native-indic-g2p/models/neural_g2p_r2_t4.pt')
checkpoint.parent.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', checkpoint, 'resume:', checkpoint.exists())

In [ ]:
import subprocess, sys
command = [
    sys.executable, 'neural_g2p.py', 'train',
    '--manifest', str(manifest_path),
    '--device', 'cuda',
    '--output', str(checkpoint),
    '--epochs', '1000',
    '--patience', '30',
    '--learning-rate', '2e-4',
    '--batch-size', '128',
    '--eval-batch-size', '256',
    '--d-model', '256',
    '--heads', '8',
    '--layers', '4',
    '--feedforward', '1024',
    '--dropout', '0.1',
    '--max-length', '384',
    '--input-mode', 'native',
]
if checkpoint.exists():
    command += ['--resume', str(checkpoint)]
subprocess.run(command, check=True)

The best checkpoint is already safe in Google Drive. Evaluate the blind split only once, after architecture and hyperparameter choices are final.

In [ ]:
# Final evaluation only — do not use this result for tuning.
# subprocess.run([sys.executable, 'neural_g2p.py', 'evaluate', '--manifest', str(manifest_path), '--device', 'cuda', '--model', str(checkpoint), '--split', 'blind'], check=True)